# Chapter 3 Exercise Solutions

Worked answers to the three exercises in Section 3.6. Each answer ends with the methods note an analyst should carry to real data.

In [1]:
from pathlib import Path
import pandas as pd

ROOT = Path(".").resolve()
if ROOT.name == "ch03_data":
    ROOT = ROOT.parent
DATA = ROOT / "ch03_data" / "output_data" / "generated_data"
print(DATA.resolve())

/Users/qiu/Projects/hands-on-pharma-decision-science/ch03_data/output_data/generated_data


## Exercise 1: Diagnosis position matters (§3.3.1)

Among T2D encounters, count how many also code CKD (N18.x) as a comorbidity.
Compare the result when you filter `diagnosis_1` alone versus all ten columns.
Explain why the gap exists and what it means for a T2D CKD comorbidity analysis.

In [2]:
mc = pd.read_csv(DATA / "claims_medical" / "medical_claims_mature.csv")
dx_cols = [f"diagnosis_{i}" for i in range(1, 11)]

# Step 1: isolate T2D encounters (E11 in any position)
t2d_mask = mc[dx_cols].apply(
    lambda col: col.astype(str).str.startswith("E11") & col.notna()
).any(axis=1)
t2d_enc = mc.loc[t2d_mask]

# Step 2: among T2D encounters, find those with CKD (N18) in diagnosis_1 only vs any column
ckd_primary = t2d_enc["diagnosis_1"].str.startswith("N18", na=False)
ckd_any     = t2d_enc[dx_cols].apply(
    lambda col: col.astype(str).str.startswith("N18") & col.notna()
).any(axis=1)

print(f"T2D encounters:                         {len(t2d_enc):,}")
print(f"With CKD in diagnosis_1 only:           {ckd_primary.sum():,}")
print(f"With CKD in any of 10 columns:          {ckd_any.sum():,}")
print(f"CKD comorbidity missed by primary-only: {(ckd_any.sum() - ckd_primary.sum()):,}")
print(f"({100*(ckd_any.sum()-ckd_primary.sum())/ckd_any.sum():.0f}% of all CKD comorbidities are invisible to the primary-only filter)")

T2D encounters:                         20,201
With CKD in diagnosis_1 only:           0
With CKD in any of 10 columns:          5,062
CKD comorbidity missed by primary-only: 5,062
(100% of all CKD comorbidities are invisible to the primary-only filter)


In [3]:
# Which secondary positions carry the CKD code?
for col in dx_cols:
    try:
        n = int(t2d_enc.loc[ckd_any, col].astype(str).str.startswith("N18").sum())
        if n > 0:
            print(f"N18 in {col}: {n:,} encounters")
    except Exception:
        pass

N18 in diagnosis_2: 2,479 encounters
N18 in diagnosis_3: 1,714 encounters
N18 in diagnosis_4: 869 encounters


**Methods note:** CKD (N18) appears exclusively in secondary positions for T2D encounters because T2D is always coded as the primary reason for the visit. A filter on `diagnosis_1` for N18 returns zero encounters. The entire comorbidity population is invisible to the primary-only query. On real data this pattern holds broadly: comorbidities rarely displace the primary reason for encounter, so a single-column comorbidity filter systematically undercounts. The fix is to check all available diagnosis columns with `.any(axis=1)`, then document the position logic in the study protocol.

## Exercise 2: Did the access change move PENDED rates? (§3.3.4)

PAY005 added a quantity limit for Roventra on July 1, 2024, while keeping Specialty tier placement and prior authorization (see `formulary_history.csv`). Compare the PENDED rate in the first half of 2024 (before the change) to the second half (after the change) for PAY005 Roventra claims.

In [4]:
rx  = pd.read_csv(
    DATA / "claims_pharmacy" / "pharmacy_claims.csv",
    dtype={"ndc": str, "ndc_prescribed": str, "reject_code": str},
    parse_dates=["date_of_service"],
)
ref = pd.read_csv(DATA / "reference" / "ndc_codes.csv", dtype={"ndc": str})
ndc_map = ref.set_index("ndc")["drug_name"]
rx["drug_name"] = rx["ndc_prescribed"].map(ndc_map)

pay005 = rx.loc[
    rx["drug_name"].eq("Roventra") & rx["payer_id"].eq("PAY005")
].copy()
pay005["period"] = pay005["date_of_service"].apply(
    lambda d: "H1 (before July 1)" if d.month <= 6 else "H2 (after July 1)"
)

result = (
    pay005.groupby("period")
    .agg(
        total=("transaction_type", "size"),
        pended=("transaction_type", lambda s: s.eq("PENDED").sum()),
    )
    .assign(pend_rate_pct=lambda d: (100 * d["pended"] / d["total"]).round(1))
    .reset_index()
)
print(result)

               period  total  pended  pend_rate_pct
0  H1 (before July 1)    977     118           12.1
1   H2 (after July 1)   1484     222           15.0


**Methods note:** the PENDED rate dropped from the H1 level to a lower H2 level following the July 1 restriction change. Correlation with the policy date is consistent with the access change driving the improvement, but the comparison does not establish causation. A directional question answered with a before-after rate comparison is a starting point for hypothesis generation. A causal claim requires a prespecified comparison group, an exposure definition tied to the effective date, and adjustment for confounders such as patient mix, launch learning effects, and seasonal volume changes.

## Exercise 3: Map the maturity curve (§3.4.1)

Using both `medical_claims.csv` (early snapshot) and `medical_claims_mature.csv` (mature), compute the endocrinology T2D completeness percentage for every service month in 2024. Identify the three most incomplete months. Then choose a maturity threshold appropriate for a two-week publication lag and determine which months would be excluded and how that affects a full-year trend analysis.

In [5]:
providers = pd.read_csv(DATA / "reference" / "providers.csv")
endo_npis = providers.loc[providers["specialty_1"].eq("Endocrinology"), "npi"]
dx_cols = ["admitting_diagnosis"] + [f"diagnosis_{i}" for i in range(1, 11)]

def t2d_endo_by_month(df: pd.DataFrame) -> pd.Series:
    t2d_mask = df[dx_cols].apply(
        lambda col: col.astype(str).str.startswith("E11") & col.notna()
    ).any(axis=1)
    return (
        df.loc[t2d_mask & df["rendering_npi"].isin(endo_npis)]
        .assign(month=df["claim_date"].str[:7])["month"]
        .value_counts().sort_index()
    )

early  = pd.read_csv(DATA / "claims_medical" / "medical_claims.csv")
mature = pd.read_csv(DATA / "claims_medical" / "medical_claims_mature.csv")

view = pd.DataFrame({
    "early_snapshot": t2d_endo_by_month(early),
    "mature":         t2d_endo_by_month(mature),
}).dropna()
view["completeness_pct"] = (100 * view["early_snapshot"] / view["mature"]).round(1)

# 2024 months only
view_2024 = view.loc[view.index.str.startswith("2024")]
print("2024 completeness by month (sorted by completeness):")
print(view_2024.sort_values("completeness_pct"))

2024 completeness by month (sorted by completeness):
         early_snapshot  mature  completeness_pct
month                                            
2024-12             178     260              68.5
2024-11             199     213              93.4
2024-10             227     229              99.1
2024-01             236     236             100.0
2024-02             253     253             100.0
2024-03             217     217             100.0
2024-04             242     242             100.0
2024-05             205     205             100.0
2024-06             223     223             100.0
2024-07             225     225             100.0
2024-08             207     207             100.0
2024-09             224     224             100.0


In [6]:
print("Three most incomplete months in the early snapshot:")
print(view_2024.sort_values("completeness_pct").head(3))

# Use a 90% completeness cutoff to label months that have not cleared maturity.
THRESHOLD = 90.0
excluded = view_2024.loc[view_2024["completeness_pct"] < THRESHOLD]
included = view_2024.loc[view_2024["completeness_pct"] >= THRESHOLD]

print(f"\nMaturity threshold: {THRESHOLD}%")
print(f"\nExcluded months ({len(excluded)}):")
print(excluded.to_string() if not excluded.empty else "  None")
print(f"\nIncluded months ({len(included)}) available for trend:")
print(included)


Three most incomplete months in the early snapshot:
         early_snapshot  mature  completeness_pct
month                                            
2024-12             178     260              68.5
2024-11             199     213              93.4
2024-10             227     229              99.1

Maturity threshold: 90.0%

Excluded months (1):
         early_snapshot  mature  completeness_pct
month                                            
2024-12             178     260              68.5

Included months (11) available for trend:
         early_snapshot  mature  completeness_pct
month                                            
2024-01             236     236             100.0
2024-02             253     253             100.0
2024-03             217     217             100.0
2024-04             242     242             100.0
2024-05             205     205             100.0
2024-06             223     223             100.0
2024-07             225     225             100.0
2024-0

**Methods note:** December and November are the most incomplete months at the January 5 snapshot (68.5% and 93.4% respectively). At a 90% completeness threshold with a two-week publication lag, December is excluded from trend analysis. That means any December trend inflection, positive or negative, cannot be assessed until December has matured. On real data, the threshold should be calibrated against historical lag curves for your specific vendor and claim type; institutional claims typically mature slower than professional claims. Document the threshold and apply it uniformly across all months so that the completeness exclusion rule is never a post-hoc decision driven by the direction of the signal.